- 集成学习:
    - 集成学习是机器学习中的一种思想,它通过多个模型组合形成一个精度更高的模型,参数组合的模型称为(弱学习器)。训练时,使用训练集依次训练出这些弱学习器,对未知的样本进行预测时,使用弱学习器进行联合预测。
    - Bagging: 随机森林 (多个模型之间相互独立,通过投票的方式进行预测)
    - Boosting: AdaBoost, GBDT , XGBoost, LightGBM (多个模型串联在一起,每个模型都依赖于前一个模型的结果,通过加权的方式进行预测)

- Bagging:
    - 有放回的的抽样,产生不同的训练集,从而训练不同的学习器.(所有树看到的特征是一样的，特征的随机性来自在“节点分裂时”，随机选一部分特征 用这些特征做划分)
    - 通过平权投票,多数表决方式决定预测结果
    - 弱学习器可以并行训练

- Boosting: 
    - 每个训练器重点关注前一个训练器不足的地方进行训练(全样本训练,但每个样本的权重不同)
    - 通过加权投票的方式,得出预测结果
    - 串行的训练方式

- 随机森林(Bagging):
    - 构建流程:
        - 随机选m条数据(有放回)随机选择k个数据特征训练决策树
        - 重复上述步骤构造n个弱决策树
        - 平权投票集成n个弱决策树

In [2]:
from sklearn.ensemble import RandomForestClassifier
model = RandomForestClassifier(
    n_estimators=100, # 决策树的数量,一般来说越多越好,但也会增加计算成本
    criterion='entropy', # 决策树的分裂标准,entropy表示信息增益,gini表示基尼系数
    random_state=42,
    max_depth=10, # 指定树的最大深度,防止过拟合,不制定则树会一直生长
    max_features='sqrt', # 每个决策树随机选择的特征数量,sqrt表示sqrt(n_features)
    bootstrap=True, # 是否采用有放回的抽样方法,False将会使用全部的训练样本
    min_samples_split=2, # 内部节点再划分所需最小样本数,默认值为2,较大的值可以防止过拟合
    min_samples_leaf=1, # 叶子节点最少样本数,默认值为1,较大的值可以防止过拟合
    # min_impurity_split=1e-7, # 内部节点再划分所需最小不纯度减少,默认值为1e-7,较大的值可以防止过拟合
    # 基尼系数小,说明节点样本标签分类基本一致
    min_impurity_decrease=1e-7, # = min_impurity_split
)

- Adaboost(Boosting):自适应提升基于Boosting思想实现的一种集成算法的核心思想是通过逐步提高那些被前一步分类错误的样本的权重来训练一个强分类器
    - 构建流程:
        - 初始化训练数据权重相同,训练第一个学习器
            - 如果有100个样本,则每个样本的初始化权重为1/100
            - 根据预测结果找一个错误率最小的分裂点,更新样本权重,模型权重
            - 错误率=错误分类数乘以权重之和
        - 根据新权值训练第二个学习器
        - 重复上面操作,训练出m个弱学习器
        - m个弱学习起预测公式=H(x)=sign(∑(α_i * h_i(x)))
            - 其中h_i(x)是第i个弱学习器的预测结果,α_i是第i个弱学习器的权重,sign函数是符号函数,当∑(α_i * h_i(x))>0时,预测结果为1,否则为-1
        - 模型权重计算公式:a_i=0.5*ln((1-ε_i)/ε_i),其中ε_i是第i个弱学习器的错误率,α_i是第i个弱学习器的权重,当ε_i越小,α_i越大,说明第i个弱学习器的预测能力越强
        - 样本权重系数计算公式:D_(i)(j)=1* exp(-α_i)(错误样本exp(a_i))
            - 预测正确权重下降,预测错误权重上升,使得后续的学习器更关注之前被错误分类的样本
        - 新权值=旧权值✖样本权重系数
        - 最终权值=新权值/Z
            - 其中Z是一个归一化因子,Z=每个元素新权值之和